# Сверка нелинейной 1D FDTD-модели МенДрайв (`./mendrive`) с линейными моделями `MenDrive_qnm*.ipynb`

Ноутбук-помощник (новый файл; исходные ноутбуки не изменяются). Структура: **Markdown-ячейка с формулами → Code-ячейка с численной проверкой (OK/FAIL и невязка)**.

Конвенции — как в `MenDrive_qnm.ipynb`, `MenDrive_qnm-mode2_je_z_TM.ipynb`, `MenDrive_qnm-mode3_jm_z_TE.ipynb`:

* временная зависимость $F(x,t)=\mathrm{Re}\,[\tilde F(x)\,e^{-i\omega t}]$ (`exp(-I*omega_t)`, ячейки 46, 308, 586);
* уравнения (ячейка 74): $\mathrm{rot}\,\vec H=-\frac{i\omega}{c}\varepsilon'\vec E+\frac{4\pi}{c}\vec j_e$, $\mathrm{rot}\,\vec E=\frac{i\omega}{c}\mu'\vec H-\frac{4\pi}{c}\vec j_m$, $\varepsilon'=\varepsilon+\frac{4\pi i\sigma}{\omega}$;
* геометрия: левая стенка $x<-a$, вакуум $-a\le x\le a$, правая (феррит) $x>a$; анзацы $e^{-iK(x+a)}$ слева и $e^{+iK(x-a)}$ справа, $\mathrm{Im}K>0$;
* 1D ($k_y=k_z=0$): TE $=(E_y,H_z)$, TM $=(E_z,H_y)$; $j_{e,y},j_{m,z}\to$ TE, $j_{e,z},j_{m,y}\to$ TM.

Комплексная амплитуда первой гармоники из FDTD (накапливается на лету в `./mendrive`, `fields_harm=1`):
$$\tilde F(x)=\frac{2}{T_{rec}}\int_{T_{rec}}F(x,t)\,e^{+i\omega t}\,dt .$$

Ядро: SageMath 9.6 или Python 3. Первая ячейка отключает препарсер Sage, дальше идёт обычный Python.

In [ ]:
try:
    preparser(False)          # SageMath: дальше -- обычный Python (float, 1j, ** и т.д.)
except NameError:
    pass                      # ядро Python 3
import os, sys, math, cmath
import numpy as np
import matplotlib.pyplot as plt

STANDALONE = '/Users/oleksiy.drozdov/articles/mendrive_llg/MenDriveCpp/standalone'
if not os.path.isdir(STANDALONE):
    STANDALONE = os.path.abspath('.')
if STANDALONE not in sys.path:
    sys.path.insert(0, STANDALONE)
import linear1d as L
import fdtd1d_ref as FD

# параметры, как в examples/linear_check_*.cfg и tests/test_fields_linear.py (код-единицы, c = 1)
GEOM = dict(N=80, a=0.2, h_l=0.2, h_r=0.2, dt_frac=0.2)
MAT  = dict(sigma_e_left=3.0, sigma_e_ferrite=0.2, eps_left=1.0, eps_ferrite=2.0)
LIN  = dict(mu_yy_lin=3.0, mu_zz_lin=3.0, sigma_m_leak=0.3)
SRC  = dict(skin_depth=0.06, src_k_phase=5.0, src_sigma_fix=1)
OMEGA, AMP = 3.0, 1.0
MFER = dict(mu_ferrite_yy=LIN['mu_yy_lin'], mu_ferrite_zz=LIN['mu_zz_lin'], sigma_m_ferrite=LIN['sigma_m_leak'])
def exact(mode, outer='pec'):
    return L.solve_linear(OMEGA, GEOM['a'], GEOM['h_l'], GEOM['h_r'], mode, amp=AMP,
                          skin_depth=SRC['skin_depth'], src_k_phase=SRC['src_k_phase'], outer=outer, **MFER, **MAT)
def verdict(ok): return 'OK' if ok else 'FAIL'
RESULTS = {}
print('STANDALONE =', STANDALONE)

## Проверка 1. Точное линейное решение удовлетворяет уравнениям Максвелла

Для $u=(E,H)$ в каждом слое $u'=A\,u+b\,j(x)$:

$$\text{TE}:\ \frac{dE_y}{dx}=i\omega\mu'_{zz}H_z-4\pi j_{m,z},\qquad \frac{dH_z}{dx}=i\omega\varepsilon' E_y-4\pi j_{e,y};$$
$$\text{TM}:\ \frac{dE_z}{dx}=-i\omega\mu'_{yy}H_y+4\pi j_{m,y},\qquad \frac{dH_y}{dx}=-i\omega\varepsilon' E_z+4\pi j_{e,z};$$

источник $\tilde j=i\,\mathrm{amp}\,e^{iK_s\xi}$, $K_s=k+i/\delta$ ($\xi$ — глубина в стенке). Граничные условия: $E=0$ на $x=-(a+h_l)$ и $x=a+h_r$ (как в FDTD), непрерывность $E,H$ при $x=\pm a$.
Критерий: относительная невязка центральных разностей $<10^{-3}$.

In [ ]:
for mode in ['je_y_left', 'je_z_left', 'jm_z_right', 'jm_y_right', 'je_y_right', 'jm_z_left']:
    f = exact(mode)
    r = L.ode_residual(f, OMEGA, GEOM['a'], GEOM['h_l'], GEOM['h_r'], mode, **MFER, **MAT)
    ok = r < 1e-3 and f.info['residual'] < 1e-9
    RESULTS['ode_' + mode] = ok
    print(f'{mode:11s} ({L.polarization_of(mode)}): невязка ОДУ = {r:.2e}, невязка СЛАУ = {f.info["residual"]:.1e}  -> {verdict(ok)}')

## Проверка 2. Дуальность TE ↔ TM

Из уравнений проверки 1 при одинаковых материалах ($\mu_{yy}=\mu_{zz}$):

$$(E_z,\;H_y)\big|_{j_{e,z}}=(E_y,\;-H_z)\big|_{j_{e,y}},\qquad (E_z,\;H_y)\big|_{j_{m,y}}=(-E_y,\;H_z)\big|_{j_{m,z}} .$$

Это проверяет согласованность знаков источников в новых режимах `je_z_*` и `jm_y_*` (в C++ — `tests/test_fields_linear.py`, п. 4).

In [ ]:
x = np.linspace(-(GEOM['a']+GEOM['h_l']), GEOM['a']+GEOM['h_r'], 801)
for te, tm, sE, sH in (('je_y_left','je_z_left',1,-1), ('jm_z_right','jm_y_right',-1,1),
                       ('je_y_right','je_z_right',1,-1), ('jm_z_left','jm_y_left',-1,1)):
    A_, B_ = exact(te)(x), exact(tm)(x)
    r = max(np.linalg.norm(B_['Ez']-sE*A_['Ey'])/np.linalg.norm(A_['Ey']),
            np.linalg.norm(B_['Hy']-sH*A_['Hz'])/np.linalg.norm(A_['Hz']))
    RESULTS['dual_'+tm] = r < 1e-10
    print(f'{te:11s} <-> {tm:11s}: {r:.2e}  -> {verdict(r < 1e-10)}')

## Проверка 3. Профили источников правой стенки в ячейке 21 (и 280) `MenDrive_qnm-mode3_jm_z_TE.ipynb`

Анзац полей в правой стенке (ячейки 48, 276, 279): $F\propto e^{+iK_r(x-a)}=e^{-iK_r(a-x)}$, $|F|=e^{-\mathrm{Im}K_r\,(x-a)}$ — затухает вглубь феррита.

В профилях `jm_ext_z_profile` (`driving_mode_3`, `_4`) и новом `jm_ext_y_profile` (`_5`) записано
$$j\propto e^{-iK_r(x+a)},\qquad |j|=e^{+\mathrm{Im}K_r\,(x+a)},\qquad \frac{|j(a)|}{|j_0|}=e^{2a\,\mathrm{Im}K_r},$$
т.е. профиль **растёт** вглубь стенки и не совпадает с «тем же скин-профилем, что и у собственной моды» (комментарий ячейки 21). Подынтегральное выражение силы $f\propto j\,D_y^*$ имеет $|f|\propto e^{\mathrm{Im}K(x+a)}e^{-\mathrm{Im}K(x-a)}=e^{2a\,\mathrm{Im}K}=\mathrm{const}$, и $\int_a^\infty$ (ячейка 332) не сходится. Предлагаемое исправление: `(x + a)` → `(a - x)`.

Числа взяты из ячейки 6 ноутбука ($A=0.5$ см, $\varepsilon_r=16$, $\mu_r=100$, $\sigma_{e,r}=1100\cdot9\cdot10^9\ \text{с}^{-1}$, $\sigma_{m,r}=0.0072\cdot9\cdot10^9$); для иллюстрации $\omega=\pi c/(2A)$, $k_y=k_z=0$. Ожидаемо: формула ноутбука — **FAIL** (растёт), исправленная — **OK** (затухает).

In [ ]:
c_sgs = 299792458*100.0
A_cm, eps_r, mu_r = 0.5, 16.0, 100.0
sig_e_r, sig_m_r = 1100*9e9, 0.0072*9e9
omega_nb = math.pi*c_sgs/(2*A_cm)
K_r = L.sqrt_im_pos((omega_nb/c_sgs)**2*(eps_r+4j*math.pi*sig_e_r/omega_nb)*(mu_r+4j*math.pi*sig_m_r/omega_nb))
h_r_nb = 0.0125*A_cm
print(f'K_r = {K_r:.4g} 1/см, скин-слой 1/Im K = {1/K_r.imag:.3e} см, h_conductor_r = {h_r_nb:.3e} см')
xs = np.array([A_cm, A_cm + 0.5*h_r_nb, A_cm + h_r_nb])
log10_nb  = (K_r.imag*(xs + A_cm))/math.log(10)      # log10|exp(-iK(x+a))|
log10_fix = (-K_r.imag*(xs - A_cm))/math.log(10)     # log10|exp(-iK(a-x))|
for xv, ln_, lf in zip(xs, log10_nb, log10_fix):
    print(f'x = {xv:.5f} см:  log10|j| (как в ноутбуке) = {ln_:9.2f},   log10|j| (a - x) = {lf:8.3f}')
grows = log10_nb[-1] > log10_nb[0]
decays = log10_fix[-1] < log10_fix[0] and abs(log10_fix[0]) < 1e-12
print('профиль ноутбука exp(-I*K*(x+a)) затухает в стенке и равен j0 на x=a:', verdict((not grows) and abs(log10_nb[0]) < 1e-12))
print('исправленный     exp(-I*K*(a-x)) затухает в стенке и равен j0 на x=a:', verdict(decays))
RESULTS['src_profile_fixed'] = decays

## Проверка 4. Разностная схема (numpy-копия `step_once`) против точного решения

`fdtd1d_ref.py` построчно повторяет линейную часть нового `step_once` (пометки `// MODE:`) и накопление гармоник `main.cpp` (`// FIELDS:`):
$$\tilde F=\frac{2\,\Delta t}{T_{rec}}\sum_k F(x,\tau_k)\,e^{+i\omega\tau_k},$$
где $\tau_k$ — истинное время отсчёта ($E$ — $t$, $H$ — $t+\Delta t/2$, $j$ — $t-\Delta t$). Сравнение **без** подбора фазы и нормы. Ошибка дискретизации $O(\Delta x)$ (ступенчатая граница), допуск 10 % при $N=80$.

Контроль конвенции: та же величина с `time_convention='plus'` равна $\tilde F^*$ и с точным решением в конвенции $e^{-i\omega t}$ **не** совпадает.

In [ ]:
NUM = {}
for mode in ['je_z_left', 'jm_z_right', 'jm_y_right', 'je_y_left']:
    sim = FD.FDTD1D(excitation_mode=mode, **GEOM, **MAT, **LIN, **SRC)
    H = FD.run_harmonics(sim, OMEGA, 40, 10, amp=AMP, ramp_periods=2.0)[1]
    NUM[mode] = (sim.xA, H)
    F = exact(mode)(sim.xA)
    En, Hn = ('Ez','Hy') if L.polarization_of(mode) == 'TM' else ('Ey','Hz')
    rE = np.linalg.norm(H[En]-F[En])/np.linalg.norm(F[En]); rH = np.linalg.norm(H[Hn+'_A']-F[Hn])/np.linalg.norm(F[Hn])
    ok = max(rE, rH) < 0.10; RESULTS['fdtd_'+mode] = ok
    print(f'{mode:11s}: rel_L2(E)={rE:.3e}  rel_L2(H)={rH:.3e}  -> {verdict(ok)}')
sim = FD.FDTD1D(excitation_mode='je_z_left', **GEOM, **MAT, **LIN, **SRC)
Hp = FD.run_harmonics(sim, OMEGA, 40, 10, amp=AMP, ramp_periods=2.0, time_convention='plus')[1]
F = exact('je_z_left')(sim.xA)
r_plus = np.linalg.norm(Hp['Ez']-F['Ez'])/np.linalg.norm(F['Ez'])
r_conj = np.linalg.norm(np.conj(Hp['Ez'])-F['Ez'])/np.linalg.norm(F['Ez'])
print(f"time_convention='plus': rel_L2 = {r_plus:.2f} (ожидаемо >> 0.1), после сопряжения {r_conj:.3e}  -> {verdict(r_plus > 0.1 and r_conj < 0.10)}")

## Проверка 5. Результат `./mendrive` (C++) против точного решения и numpy-копии

Нужен собранный `./mendrive` (после установки Xcode Command Line Tools: `xcode-select --install`, затем `make`):
```sh
./mendrive -c examples/linear_check_TM.cfg      # и/или linear_check_TE.cfg
```
Критерии: C++ против точного решения (N=160) $<5\%$ без подбора фазы и нормы; снимок по фазе $\varphi$ равен $\mathrm{Re}[\tilde F e^{-i\omega t_\varphi}]$ (с $t+\Delta t/2$ для $H$) с точностью $10^{-3}$. Если файлов нет, проверка помечается SKIP.

In [ ]:
def check_cpp(run_dir, mode):
    p = os.path.join(STANDALONE, run_dir, 'fields_harm_n1.csv')
    if not os.path.exists(p):
        print(f'{run_dir}: нет {p} -> SKIP (сначала make && ./mendrive -c examples/{os.path.basename(run_dir)}.cfg)'); return None
    d = L.read_profiles_csv(p)
    lp = L.analytic_for_run(os.path.join(STANDALONE, run_dir), n_points=2001)
    li = L.read_profiles_csv(lp)
    En, Hn = ('Ez','Hy') if L.polarization_of(mode) == 'TM' else ('Ey','Hz')
    out = []
    for nm in (En, Hn):
        r, alpha, *_ = L.compare_profiles(li['x'], L.complex_column(li, nm), d['x'], L.complex_column(d, nm), norm='none', fit='none')
        out.append(r); print(f'{run_dir}: {nm}: rel_L2 (без подбора) = {r:.3e}')
    ok = max(out) < 0.05
    sp = os.path.join(STANDALONE, run_dir, 'fields_snap.csv')
    if os.path.exists(sp):
        sn = L.read_profiles_csv(sp); head = open(sp, encoding='utf-8').readline()
        dt = float(head.split('dt=')[1].split()[0]); w = float(head.split('omega0=')[1].split()[0])
        worst = 0.0
        for ph in sorted(set(sn['phase_target'])):
            m = sn['phase_target'] == ph; t = sn['t'][m][0]
            e1 = np.max(np.abs(sn[En][m] - (L.complex_column(d, En)*np.exp(-1j*w*t)).real))/np.max(np.abs(sn[En][m]))
            e2 = np.max(np.abs(sn[Hn][m] - (L.complex_column(d, Hn)*np.exp(-1j*w*(t+0.5*dt))).real))/np.max(np.abs(sn[Hn][m]))
            worst = max(worst, e1, e2)
        print(f'{run_dir}: снимки vs Re[F~ exp(-i w t)]: {worst:.2e}'); ok = ok and worst < 1e-3
    print(f'{run_dir}: -> {verdict(ok)}'); return ok
RESULTS['cpp_TM'] = check_cpp('out/linear_check_TM', 'je_z_left')
RESULTS['cpp_TE'] = check_cpp('out/linear_check_TE', 'jm_z_right')

## Графики в стиле ноутбуков `MenDrive_qnm*` (ячейки 521–537)

Re, Im, Abs, Arg/π тангенциальных компонент $E$ и $H$ по трём участкам: левая стенка $[-a-h_l,-a]$, вакуум $[-a,a]$, правая стенка $[a,a+h_r]$. TM показан зелёным (`color="green"`), TE — синим. Сплошная линия — точное линейное решение, пунктир — FDTD (numpy-копия схемы или C++, если файл есть).

In [ ]:
def notebook_style(mode, xs_num, F_num, title_extra=''):
    a, hl, hr = GEOM['a'], GEOM['h_l'], GEOM['h_r']
    En, Hn = ('Ez','Hy') if L.polarization_of(mode) == 'TM' else ('Ey','Hz')
    col = 'green' if L.polarization_of(mode) == 'TM' else 'tab:blue'
    f = exact(mode)
    segs = [(-a-hl, -a), (-a, a), (a, a+hr)]
    for nm, what in ((En, 'электрического поля'), (Hn, 'напряженности магнитного поля H')):
        fig, ax = plt.subplots(2, 2, figsize=(12, 7.5))
        for axk, (pn, fn) in zip(ax.flat, (('Re', np.real), ('Im', np.imag), ('Abs', np.abs), ('Arg/pi', lambda z: np.angle(z)/np.pi))):
            for (x0, x1) in segs:
                xx = np.linspace(x0, x1, 200)
                axk.plot(xx, fn(f(xx)[nm]), color=col, lw=1.8)
            axk.plot(xs_num, fn(F_num[nm]), 'k--', lw=0.9, label='FDTD')
            axk.set_title(f'График {pn} тангенциальной компоненты {what}', fontsize=9); axk.grid(True)
            for xv in (-a, a): axk.axvline(xv, color='gray', ls=':', lw=0.8)
        ax.flat[0].legend(fontsize=8); fig.suptitle(f'{mode}: {nm} {title_extra}'); fig.tight_layout(); plt.show()

xs_, H_ = NUM['je_z_left']; notebook_style('je_z_left', xs_, {'Ez': H_['Ez'], 'Hy': H_['Hy_A']}, '(numpy-копия схемы, N=80)')
xs_, H_ = NUM['jm_z_right']; notebook_style('jm_z_right', xs_, {'Ey': H_['Ey'], 'Hz': H_['Hz_A']}, '(numpy-копия схемы, N=80)')

## Шаг 6. Профили из вашей линейной модели (`MenDrive_qnm*.ipynb`) → CSV → сравнение с C++

**(а) Выгрузка.** В КОНЦЕ нужного ноутбука (после ячейки `subs_fields(...)` ≈ 506, где определены `Ey_left_conductor_d`, …) выполните:

```python
import sys; sys.path.insert(0, '/Users/oleksiy.drozdov/articles/mendrive_llg/MenDriveCpp/standalone')
from linear1d import export_sage_profiles
export_sage_profiles('/Users/oleksiy.drozdov/articles/mendrive_llg/MenDriveCpp/standalone/linear_mode3.csv',
    {'Ey': (Ey_left_conductor_d, Ey_vacuum_d, Ey_right_conductor_d),
     'Hz': (Hz_left_conductor_d, Hz_vacuum_d, Hz_right_conductor_d),
     'Ez': (Ez_left_conductor_d, Ez_vacuum_d, Ez_right_conductor_d),
     'Hy': (Hy_left_conductor_d, Hy_vacuum_d, Hy_right_conductor_d)},
    A=A, h_l=h_conductor_l, h_r=h_conductor_r, y0=0)
```
(значения в точке $y=y_0$; для `driving_mode_1/_4/_5` $k_y=0$ и `phase_y`$=1$).

**(б) Сравнение.** Геометрия ноутбука ($A=0.5$ см, тонкие стенки) и FDTD обычно различаются, поэтому сравнение ведётся в $\xi=x/a$, с нормировкой по $\max|F|$ и подбором общей фазы $\alpha$:
$$\mathrm{rel}_{L2}=\frac{\lVert \alpha\,\hat F_{C++}-\hat F_{lin}\rVert}{\lVert \hat F_{lin}\rVert},\qquad \hat F=\frac{F}{\max|F|},\qquad \alpha=e^{i\arg\langle \hat F_{C++},\hat F_{lin}\rangle}.$$
Критерий OK: $<5\%$. Для совпадения по абсолютной шкале переведите параметры в код-единицы (README §10): $\omega_{code}=\omega L_0/c$, $\sigma_{code}=\sigma L_0/c$.

In [ ]:
LIN_CSV = os.path.join(STANDALONE, 'linear_mode3.csv')        # из шага (а)
CPP_DIR = os.path.join(STANDALONE, 'out/mode_jm_z_right')       # прогон ./mendrive с fields_harm=1
p_cpp = os.path.join(CPP_DIR, 'fields_harm_n1.csv')
if os.path.exists(LIN_CSV) and os.path.exists(p_cpp):
    li, nl = L.read_profiles_csv(LIN_CSV), L.read_profiles_csv(p_cpp)
    a_lin = float(open(LIN_CSV, encoding='utf-8').readline().split('a=')[1].split()[0])
    a_cpp = float([l for l in open(p_cpp, encoding='utf-8') if l.startswith('#')][0].split(' a=')[1].split()[0])
    for nm in ('Ey', 'Hz', 'Ez', 'Hy'):
        if nm + '_re' not in li or nm + '_re' not in nl: continue
        u, v = L.complex_column(li, nm), L.complex_column(nl, nm)
        if np.nanmax(np.abs(u)) == 0 or np.nanmax(np.abs(v)) == 0: continue
        r, alpha, xx, uu, vv = L.compare_profiles(li['x']/a_lin, u, nl['x']/a_cpp, v, norm='max', fit='phase')
        RESULTS['user_'+nm] = r < 0.05
        print(f'{nm}: rel_L2 = {r:.3e}, общая фаза = {np.angle(alpha)/np.pi:+.4f} pi  -> {verdict(r < 0.05)}')
        plt.figure(figsize=(10, 3.6))
        for k, (pn, fn) in enumerate((('Abs', np.abs), ('Re', np.real), ('Im', np.imag))):
            plt.subplot(1, 3, k+1); plt.plot(xx, fn(uu), 'k', lw=2, alpha=0.6, label='линейная')
            plt.plot(xx, fn(vv), '--', label='C++'); plt.title(f'{pn} {nm}'); plt.grid(True); plt.xlabel('x/a')
        plt.legend(fontsize=8); plt.tight_layout(); plt.show()
else:
    print('SKIP: нет', LIN_CSV if not os.path.exists(LIN_CSV) else p_cpp)

## Итог

In [ ]:
for k, v in RESULTS.items():
    print(f'{k:22s} {"SKIP" if v is None else verdict(v)}')
print('\nВСЕ ВЫПОЛНЕННЫЕ ПРОВЕРКИ:', verdict(all(v for v in RESULTS.values() if v is not None)))